# RFAntibody v1.0
*Last updated: 19 Feb 2025*

Welcome to RFAntibody! This notebook includes setup, input file formatting, and all 3 steps of the RFAntibody pipeline. Use the outline (first button on the left sidebar) to navigate quickly between sections of the notebook.

Note that you'll need to run this with a GPU (`Runtime > Change runtime type > H100 GPU`). An academic email can get you access to Colab Pro which allows access to better GPUs.

Sections with  **- TO EDIT** denoted require your input. For these sections, look for `# TODO` comments that indicate what needs to be edited. Most other cells can be run without modifications.

# Setup

### A. Checks

#### A.1. Check colab environment

In [ ]:
# @title
%%bash
echo "=== OS / Python ==="
uname -a || true
lsb_release -a || cat /etc/os-release || true
python --version
which python
echo

echo "=== GPU ==="
command -v nvidia-smi >/dev/null && nvidia-smi || \
  echo "nvidia-smi not found. In Colab: Runtime → Change runtime type → GPU"

=== OS / Python ===
Linux de79f88de1bf 6.6.113+ #1 SMP Mon Feb  2 12:27:57 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.5 LTS
Release:	22.04
Codename:	jammy
Python 3.12.12
/usr/local/bin/python

=== GPU ===
Thu Mar  5 06:31:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |     

No LSB modules are available.


In [ ]:
# @title
import sys, platform

print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("\nTorch:", torch.__version__)
    print("Torch CUDA built:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("Capability:", torch.cuda.get_device_capability(0))
except Exception as e:
    print("\nPyTorch not importable yet (that's okay). Error:")
    print(" ", repr(e))


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35

Torch: 2.10.0+cu128
Torch CUDA built: 12.8
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Capability: (8, 0)


#### A.2. Check packages

In [ ]:
# @title
%%bash
set -euo pipefail

sudo apt-get update -y

# Core tools: git, downloaders, build toolchain, and utilities that often prevent headaches
sudo apt-get install -y \
  git \
  wget \
  curl \
  ca-certificates \
  build-essential \
  cmake \
  pkg-config \
  rsync \
  unzip \
  zip \
  tree \
  less

echo "Done installing system packages."


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,388 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntu

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 6.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


### B. Set up Google Drive

#### B.1. Configuration

In [ ]:
# @title
# Where the repo will live in the Colab runtime:
WORKDIR = "/content"
REPO_DIR = f"{WORKDIR}/RFantibody"

# If you want persistent caching in Google Drive (recommended):
USE_DRIVE_CACHE = True

# Where to cache RFantibody weights in Drive (persistent):
DRIVE_CACHE_DIR = "/content/drive/MyDrive/RFantibody_cache"

# pin a specific commit/tag for reproducibility (recommended).
# Leave as None to use the default branch HEAD.
PIN_REF = "2b864664e48c87edac7f119c157f77e5bbad16e9"  # latest as of 28 Jan 2026

#### B.2. Mount Google Drive
Prevents you from having to re-download everything if your Colab notebook restarts.

In [ ]:
# @title
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Skipping Drive mount (USE_DRIVE_CACHE=False).")


Mounted at /content/drive


### C. Set up RFAntibody

#### C.1. Clone RFAntibody github repository

In [ ]:
# @title
import os, subprocess, textwrap, pathlib

os.makedirs(WORKDIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", "https://github.com/RosettaCommons/RFantibody.git", REPO_DIR])
else:
    print(f"Repo already exists at {REPO_DIR} — skipping clone.")

# clones a specific commit (for reproducibility - see above to set PIN_REF)
if PIN_REF:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "--all", "--tags"])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", PIN_REF])

# Show repo status
subprocess.check_call(["git", "-C", REPO_DIR, "rev-parse", "HEAD"])
subprocess.check_call(["git", "-C", REPO_DIR, "status", "--porcelain=v1"])


0

#### C.2. Download model weights
The first time you download should take about 2 min. We'll cache just the model weights in Google Drive so that you don't need to redownload them each time the notebook restarts (saves time and allows for reproducibility).

In [ ]:
# @title
import os, subprocess, pathlib, shutil

# Ensure that download script exists
repo = pathlib.Path(REPO_DIR)
download_script = repo / "include" / "download_weights.sh"
assert download_script.exists(), f"Missing: {download_script}"

# Decide cache location
if USE_DRIVE_CACHE:
    cache_root = pathlib.Path(DRIVE_CACHE_DIR)
else:
    cache_root = pathlib.Path("/content/RFantibody_cache")

cache_root.mkdir(parents=True, exist_ok=True)

# Snapshot existing directories in your GDrive before download
before = {p.name for p in repo.iterdir() if p.is_dir()}

# Download weights, if you don't have them cached already
src = repo / "weights"
dst = cache_root / "weights"
if dst.exists():
    print(f"Cache already has weights at {dst} — skipping download.")
else:
  # download weights
  p = subprocess.run(
    ["bash", str(download_script)],
    cwd=str(repo),
    text=True,
    capture_output=True
  )
  print(p.stdout)
  print(p.stderr)
  p.check_returncode()
  # look at what directories were created
  after = {p.name for p in repo.iterdir() if p.is_dir()}
  new_dirs = sorted(list(after - before))
  print("\nNew directories created by download script:", new_dirs)
  # cache weights
  if USE_DRIVE_CACHE:
    print(f"Caching weights to {dst}...")
    shutil.copytree(src, dst)

# Now ensure repo points to cached copies (symlink if possible)
# We'll replace repo/weights with symlink to cache_root/weights when Drive
# caching is used.
if USE_DRIVE_CACHE:
    if not src.is_symlink() and src.exists():
      # remove the local copy to avoid doubling disk
      print(f"Replacing local {src} with symlink to cache...")
      if src.is_dir():
          shutil.rmtree(src)
      else:
          src.unlink()
      os.symlink(str(dst), str(src))
      print(f"Symlinked {src} -> {dst}")

print("\nDone. Repo top-level dirs now include:")
for p in sorted(repo.iterdir()):
    tag = "-> " + os.readlink(p) if p.is_symlink() else ""
    print(f" - {p.name} {'(symlink)' if p.is_symlink() else ''} {tag}")


Cache already has weights at /content/drive/MyDrive/RFantibody_cache/weights — skipping download.

Done. Repo top-level dirs now include:
 - .git  
 - .gitignore  
 - CLAUDE.md  
 - Dockerfile  
 - LICENSE  
 - README.md  
 - github_image.png  
 - include  
 - pyproject.toml  
 - rfantibody.def  
 - scripts  
 - src  
 - test  
 - uv.lock  


#### C.3. Patch weights references
This will hopefully be fixed in the repo for later versions of this notebook

RFDiffusion

In [ ]:
# @title
from pathlib import Path
import re, shutil

# dst should already exist from your caching block (e.g., cache_root/"weights")
dst = Path(dst)  # ensure Path
new_ckpt = dst / "RFdiffusion_Ab.pt"

assert new_ckpt.exists(), f"Expected checkpoint not found: {new_ckpt}"

mr = Path(REPO_DIR) / "src" / "rfantibody" / "rfdiffusion" / "inference" / "model_runners.py"
assert mr.exists(), f"Missing file to patch: {mr}"

# Backup once (won't overwrite if already backed up)
bak = mr.with_suffix(mr.suffix + ".bak")
if not bak.exists():
    shutil.copy2(mr, bak)
    print(f"Backed up -> {bak}")

txt = mr.read_text()

# Replace the hard-coded ckpt assignment line(s)
pattern = r"(self\.ckpt_path\s*=\s*)['\"][^'\"]*base_complex_finetuned_BFF_9\.pt['\"]"
replacement = r"\1" + repr(str(new_ckpt))  # writes a proper Python string literal

txt2, n = re.subn(pattern, replacement, txt)

if n == 0:
    # If it didn't match, show nearby lines to help you locate the actual assignment
    raise RuntimeError(
        "Did not find expected ckpt_path assignment for base_complex_finetuned_BFF_9.pt.\n"
        "Search for 'ckpt_path' in model_runners.py and adjust the regex accordingly."
    )

mr.write_text(txt2)
print(f"Patched {mr} (replaced {n} occurrence(s))")
print(f"Checkpoint path now set to: {new_ckpt}")


Backed up -> /content/RFantibody/src/rfantibody/rfdiffusion/inference/model_runners.py.bak
Patched /content/RFantibody/src/rfantibody/rfdiffusion/inference/model_runners.py (replaced 1 occurrence(s))
Checkpoint path now set to: /content/drive/MyDrive/RFantibody_cache/weights/RFdiffusion_Ab.pt


ProteinMPNN

In [ ]:
# @title
from pathlib import Path
import re, shutil

dst = Path(dst)  # your cache weights dir
new_ckpt = dst / "ProteinMPNN_v48_noise_0.2.pt"
assert new_ckpt.exists(), f"Missing expected weight file: {new_ckpt}"

script = Path(REPO_DIR) / "scripts" / "proteinmpnn_interface_design.py"
assert script.exists(), f"Missing script: {script}"

# backup
bak = script.with_suffix(script.suffix + ".bak")
if not bak.exists():
    shutil.copy2(script, bak)
    print(f"Backed up -> {bak}")

txt = script.read_text()

# Replace that broken join(...) line (be tolerant to whitespace)
pattern = r"default_ckpt\s*=\s*os\.path\.join\(\s*os\.path\.dirname\(__file__\)\s*,\s*['\"]/home/weights/ProteinMPNN_v48_noise_0\.2\.pt['\"]\s*\)"
replacement = f"default_ckpt = {repr(str(new_ckpt))}"

txt2, n = re.subn(pattern, replacement, txt)
if n == 0:
    raise RuntimeError("Did not find the expected default_ckpt os.path.join(...) line to patch.")

script.write_text(txt2)
print(f"Patched {script} (replaced {n} occurrence(s))")
print(f"default_ckpt -> {new_ckpt}")

Backed up -> /content/RFantibody/scripts/proteinmpnn_interface_design.py.bak
Patched /content/RFantibody/scripts/proteinmpnn_interface_design.py (replaced 1 occurrence(s))
default_ckpt -> /content/drive/MyDrive/RFantibody_cache/weights/ProteinMPNN_v48_noise_0.2.pt


RF2

In [ ]:
# @title
from pathlib import Path
import shutil
import re

dst = Path(dst)
new_ckpt = dst / "RF2_ab.pt"

assert new_ckpt.exists(), f"Expected RF2 checkpoint not found: {new_ckpt}"

yaml_path = Path(REPO_DIR) / "src" / "rfantibody" / "rf2" / "config" / "base.yaml"
assert yaml_path.exists(), f"Missing config file: {yaml_path}"

# Backup once
bak = yaml_path.with_suffix(".yaml.bak")
if not bak.exists():
    shutil.copy2(yaml_path, bak)
    print(f"Backed up -> {bak}")

txt = yaml_path.read_text()

# Replace the weights path
pattern = r"(model_weights:\s*)/home/weights/RF2_ab\.pt"
replacement = r"\1" + str(new_ckpt)

txt2, n = re.subn(pattern, replacement, txt)

if n == 0:
    raise RuntimeError(
        "Did not find expected model_weights line with /home/weights/RF2_ab.pt"
    )

yaml_path.write_text(txt2)

print(f"Patched {yaml_path} (replaced {n} occurrence(s))")
print(f"model_weights now set to: {new_ckpt}")

Backed up -> /content/RFantibody/src/rfantibody/rf2/config/base.yaml.bak
Patched /content/RFantibody/src/rfantibody/rf2/config/base.yaml (replaced 1 occurrence(s))
model_weights now set to: /content/drive/MyDrive/RFantibody_cache/weights/RF2_ab.pt


#### C.4. Set up & test Python environment

In [ ]:
# @title
# install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh

# Add uv to PATH for this notebook session
import os
os.environ["PATH"] += ":/root/.local/bin"

!uv --version


downloading uv 0.10.8 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
uv 0.10.8


In [ ]:
# @title
# Run uv sync inside the repo directory
# Note it's normal for this to take a while
subprocess.check_call(["uv", "sync"], cwd=REPO_DIR)

print("uv sync completed")


uv sync completed


In [ ]:
# @title
# check that CUDA works from inside the environment
output = subprocess.check_output([
    "uv", "run", "python", "-c",
    """
import torch
print("Torch version:", torch.__version__)
print("Built with CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
"""
], cwd=REPO_DIR, text=True)

print(output)


Torch version: 2.3.1+cu118
Built with CUDA: 11.8
CUDA available: True
GPU name: NVIDIA A100-SXM4-80GB



In [ ]:
# @title
# check that RFantibody CLI tools exist here by printing their help manuals
print(subprocess.check_output(["uv", "run", "rfdiffusion", "--help"], cwd=REPO_DIR, text=True))
print(subprocess.check_output(["uv", "run", "proteinmpnn", "--help"], cwd=REPO_DIR, text=True))
print(subprocess.check_output(["uv", "run", "rf2", "--help"], cwd=REPO_DIR, text=True))

Usage: rfdiffusion [OPTIONS]

  Run RFdiffusion antibody design.

  Generates antibody structures targeting a specific antigen using diffusion-
  based design. The designed loops can be specified with length ranges.

  Examples:
      # Basic antibody design
      rfdiffusion -t antigen.pdb -f framework.pdb -o my_designs/ab

      # Design with specific loop lengths and hotspots
      rfdiffusion -t antigen.pdb -f framework.pdb -l "H1:7,H3:5-13" -h "A305" -n 5

      # Output to Quiver file
      rfdiffusion -t antigen.pdb -f framework.pdb -q designs.qv -n 100

Options:
  -t, --target PATH          Target PDB file (antigen)  [required]
  -f, --framework PATH       Framework PDB file (antibody scaffold)
                             [required]
  -o, --output PATH          Output prefix for designs (default:
                             designs/ab_des)
  -q, --output-quiver PATH   Output to Quiver file instead of PDB files
  -n, --num-designs INTEGER  Number of designs to generate (defaul

# Prepare input files
The antibody-finetuned version of RFdiffusion in RFantibody requires an HLT-remarked framework structure as input.

We HIGHLY recommend mounting your Google Drive (if you chose not to do so above) and uploading everything to a folder in your Google Drive to avoid losing all work if the runtime ends, runs out of units, etc.

## A. TO EDIT - Prepare antibody input pdb

The following requires a Chothia numbered antibody PDB. For computational efficiency, we recommend that you crop to just the VH + VL if you are designing an Fv, or crop to just VH if you are designing a nanobody.

The recommended method to do this:
1. Search for your desired PDB structure on [SAbDab](https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab) and download the Chothia numbered structure.
2. Either: **(a)** open the pdb file in VSCode, vim, etc and delete all constant regions after VH/VL (or after VHH), or **(b)** open the pdb file in PyMOL, select only VH+VL (or only VHH), copy selection to new object, de-select original pdb in right sidebar, export current molecule as pdb.

You can then use the cell below on the cleaned pdb to format the file correctly for use with RFAntibody. The formatted file will be stored in the same place as your uploaded file, with the `_HLT.pdb` suffix added.

In [ ]:
# @title
%pip install biotite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 103.1 MB/s eta 0:00:00


In [ ]:
cleaned_ab_pdb = "/content/drive/MyDrive/hackathon_RFantibody/1n8z_chothia_cleaned.pdb" # TODO: YOUR AB PDB HERE
heavy_chid = "B" # TODO: YOUR ACTUAL HEAVY CHAIN ID
light_chid = "A" # TODO: YOUR ACTUAL LIGHT CHAIN ID

final_ab_pdb = cleaned_ab_pdb.replace(".pdb", "_HLT.pdb")

# run conversion script
p = subprocess.run(["python", f"scripts/util/chothia2HLT.py",
                    "--heavy", heavy_chid, "--light", light_chid, cleaned_ab_pdb],
               cwd=REPO_DIR, text=True, capture_output=True)
print("STDOUT:\n", p.stdout)
print("STDERR:\n", p.stderr)
p.check_returncode()


STDOUT:
 
STDERR:
 


## B. TO EDIT - Prepare target input pdb
Upload your single- or multi-chain target pdb, then use the cells below to format your target (edit to the correct chain id "T" and renumber residues). You may want to crop your target for efficiency. Read more about cropping [here](https://github.com/RosettaCommons/RFantibody/tree/main?tab=readme-ov-file#truncating-your-target-protein).

The formatted target pdb will be created in the same folder as your uploaded target pdb, with the suffix `_HLT.pdb` added.

In [ ]:
# @title
# (code adapted from chothia2HLT.py)

import biotite.structure.io as io
import biotite.structure as struc
from biotite.structure.io.pdb import PDBFile
import numpy as np

def edit_target_pdb(target_pdb, target_chids):
  aa = io.load_structure(target_pdb)

  # Subset the structure to only include protein chains
  protein_residues = [
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY",
    "HIS", "ILE", "LEU", "LYS", "MET", "PHE", "PRO", "SER",
    "THR", "TRP", "TYR", "VAL"
  ]
  protein_mask = np.isin(aa.res_name, protein_residues)
  aa = aa[protein_mask]

  new_atoms = []
  current_residue = 1
  for t in target_chids:
    chain_mask = aa.chain_id == t
    if chain_mask.sum() != 0:
        chain_atoms = aa[chain_mask]
        # Rename chain to T
        chain_atoms.chain_id = np.full(len(chain_atoms), "T")
        # Renumber residues to prevent clashes from multiple chains
        for res in set(chain_atoms.res_id):
            res_mask = chain_atoms.res_id == res
            res_atoms = chain_atoms[res_mask]
            res_atoms.res_id = np.full(len(res_atoms), current_residue)
            current_residue += 1
            for atom in res_atoms:
              new_atoms.append(atom)

  # Create the new AtomArray and save it
  new_aa = PDBFile()
  new_aa.set_structure(struc.array(new_atoms))

  new_filename = target_pdb.replace(".pdb", "_HLT.pdb")
  new_aa.write(new_filename)
  print(f"Wrote formatted target to {new_filename}")

  return new_filename


In [ ]:
target_pdb = "/content/drive/MyDrive/hackathon_RFantibody/6d8b_crop.pdb" # TODO: YOUR ACTUAL TARGET PDB
target_chids = ["A","B","C","D","E","F"] # TODO: YOUR ACTUAL TARGET CHAINS (keep list format even if just 1 chain)

final_target_pdb = edit_target_pdb(target_pdb, target_chids)


Wrote formatted target to /content/drive/MyDrive/hackathon_RFantibody/6d8b_crop_HLT.pdb


# Run pipeline

## A. RFdiffusion
The first step in RFantibody is to generate antibody-target docks using an antibody-finetuned version of RFdiffusion.

### A.1. Run RFdiffusion - TO EDIT
We HIGHLY recommend mounting your Google Drive (if you chose not to do so above) and setting all output paths to a folder in your Google Drive to avoid losing all work if the runtime ends, runs out of units, etc.

Fill in the cells below with the following options:
* `target`: path to your formatted target pdb (only change if needed; by default we'll use the path generated in the cells above)
* `framework`: path to your formatted antibody pdb. RFdiffusion will only design the structure and sequence of regions of the framework which are annotated as loops, this allows us to design the dock and loops of already optimized frameworks.
* `output`: The prefix of the .pdb file outputs that we will generate.
* `num_designs`: The number of designs we should generate.
* `design_loops`: A mapping of each CDR loop to a range of allowed loop lengths. The length of each loop is sampled uniformly from this range and is sampled independently of the lengths sampled for other loops. If a CDR loop exists in the framework but is not specified, this CDR loop will have its sequence and structure fixed during design. If a CDR loop is included but no range of lengths is provided (e.g., H1:7), this CDR loop will have its sequence and structure designed but only with the specified length.
* `hotspots`: A list of hotspot residues that define our epitope. More information on selecting hotspots in more detail can be found [here](https://github.com/RosettaCommons/RFantibody/tree/main?tab=readme-ov-file#selecting-a-target-site). **NB: make sure you're specifying hotspot residues on your newly renumbered target file - not the original file!**

In [ ]:
target = final_target_pdb
framework = final_ab_pdb
output = "/content/drive/MyDrive/RFantibody_example_run/rfabd_test_out/rfabd_test_6d8b" # TODO: YOUR DESIRED OUTPUT DIR (Google Drive recommended!!!)
num_designs = "3" # TODO: DESIRED NUMBER OF DESIGNS TO OUTPUT
design_loops = "H1:7,H2:7-8,H3:5-17,L1:7,L2:7-8,L3:7-12" # TODO: DESIRED CDR LOOP LENGTHS
hotspots = "T238,T254,T300" # TODO: HOTSPOTS DETERMINED FROM YOUR *CLEANED* TARGET PDB  ### To RB: these aren't hydrophobic, but they're based on actual hotspots in a different flu strain in 3gbm, which has an antibody around the truncation. i just chose these to get an animation

Then run the following cell to run RFdiffusion (took ~2 minutes for 1 design on a H100 gpu).

In [ ]:
# @title
import subprocess

cmd = ["uv", "run", "rfdiffusion",
       "-t", target, "-f", framework,
       "-o", output, "-n", num_designs,
       "-l", design_loops, "-h", hotspots]

p = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
print("STDOUT:\n", p.stdout)
print("STDERR:\n", p.stderr)

STDOUT:
 Running RFdiffusion with 3 designs...
Target: /content/drive/MyDrive/hackathon_RFantibody/6d8b_crop_HLT.pdb
Framework: /content/drive/MyDrive/hackathon_RFantibody/1n8z_chothia_cleaned_HLT.pdb
Design loops: H1:7,H2:7-8,H3:5-17,L1:7,L2:7-8,L3:7-12
Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
[2026-03-05 07:09:08,087][rfantibody.rfdiffusion.inference.model_runners][INFO] - Reading checkpoint from /content/drive/MyDrive/RFantibody_cache/weights/RFdiffusion_Ab.pt
This is inf_conf.ckpt_path
/content/drive/MyDrive/RFantibody_cache/weights/RFdiffusion_Ab.pt
Assembling -model, -diffuser and -preprocess configs from checkpoint
USING MODEL CONFIG: self._conf[model][n_extra_block] = 4
USING MODEL CONFIG: self._conf[model][n_main_block] = 32
USING MODEL CONFIG: self._conf[model][n_ref_block] = 4
USING MODEL CONFIG: self._conf[model][d_m

In [ ]:
# @title
# check if the run succeeded, print error if failed
if "Error" not in p.stderr:
  print(f"Run succeeded! Outputs can be found in {output}*")
else:
  print(f"Encountered an error during run. STDERR:")
  print(p.stderr)

Run succeeded! Outputs can be found in /content/drive/MyDrive/RFantibody_example_run/rfabd_test_out/rfabd_test_6d8b*


If your run succeeded, you should now have pdb files for each of your outputs at the path you specified in the `output` variable. We will use these files to run the next step, ProteinMPNN, to design the CDR sequences.

### A.2. Visualize outputs - TO EDIT

In [ ]:
# @title
# --- Inline PDB viewer with chain coloring + CDR highlighting (from REMARK PDBinfo-LABEL) ---

from pathlib import Path
import re

def _parse_cdrs_from_pdb_remarks(pdb_text: str):
    """
    Parse lines like:
      REMARK PDBinfo-LABEL:  106 H3
    Returns: dict(chain -> sorted list of residue numbers), e.g. {"H":[...], "L":[...]}
    Assumption: labels start with H or L (e.g., H1/H2/H3, L1/L2/L3), and residue numbers
    correspond to that chain's residue numbering in the PDB.
    """
    cdr_by_chain = {"H": set(), "L": set()}
    pat = re.compile(r"^REMARK\s+PDBinfo-LABEL:\s+(\d+)\s+([HL]\d)\s*$")

    for line in pdb_text.splitlines():
        m = pat.match(line)
        if not m:
            continue
        resi = int(m.group(1))
        label = m.group(2)  # e.g., H3
        chain = label[0]    # "H" or "L"
        cdr_by_chain[chain].add(resi)

    # drop empty chains
    cdr_by_chain = {k: sorted(v) for k, v in cdr_by_chain.items() if v}
    return cdr_by_chain


def view_pdb_with_cdrs(
    pdb_path,
    width=900,
    height=550,
    spin=False,
    show_ligands_as_sticks=True,
):
    """
    Colors chains:
      T = teal, H = pink, L = wheat
    Highlights CDR residues (as parsed from REMARK PDBinfo-LABEL) in red.
    """
    # Lazy install if needed
    try:
        import py3Dmol
    except ImportError:
        import sys, subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "py3Dmol"])
        import py3Dmol

    pdb_path = Path(pdb_path)
    if not pdb_path.exists():
        raise FileNotFoundError(f"Could not find PDB file: {pdb_path}")

    pdb_text = pdb_path.read_text()
    cdr_by_chain = _parse_cdrs_from_pdb_remarks(pdb_text)

    v = py3Dmol.view(width=width, height=height)
    v.addModel(pdb_text, "pdb")

    # Color map
    map = {"T": "teal", "H": "pink", "L": "wheat"}
    v.setStyle({}, {"cartoon": {"colorscheme": {"prop": "chain", "map": map}}})

    v.zoomTo()
    if spin:
        v.spin(True)

    # Helpful info in notebook output (optional)
    if cdr_by_chain:
        print("CDR residues found (from REMARK PDBinfo-LABEL):")
        for ch, resis in cdr_by_chain.items():
            print(f"  Chain {ch}: {len(resis)} residues (e.g., {resis[:10]}{'...' if len(resis)>10 else ''})")
    else:
        print("No CDR REMARK lines found matching: 'REMARK PDBinfo-LABEL:  <resi> <H/L><1-3>'")

    return v.show()

In [ ]:
PDB_PATH = "/content/drive/MyDrive/RFantibody_example_run/rfabd_test_out/rfabd_test_6d8b_1.pdb" # TODO: PATH TO STRUCTURE
view_pdb_with_cdrs(PDB_PATH, spin=False)


CDR residues found (from REMARK PDBinfo-LABEL):
  Chain H: 30 residues (e.g., [26, 27, 28, 29, 30, 31, 32, 52, 53, 54]...)
  Chain L: 22 residues (e.g., [150, 151, 152, 153, 154, 155, 156, 172, 173, 174]...)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## B. ProteinMPNN - TO EDIT
The second step in RFantibody is to take the docks generated by RFdiffusion and assign sequences to the CDR loops. We do this using the base version of ProteinMPNN, ie. not an antibody-finetuned model.

Fill out the following options in the cells below:
* `input_dir`: Input directory of PDB files from RFDiffusion
* `output_dir` Output directory for PDB files generated by MPNN
* `loops`: Loops to design (default: H1,H2,H3,L1,L2,L3)
* `seqs_per_struct`: Sequences MPNN should generate per diffusion structure
* `temp`: Sampling temperature (default: 0.1, higher = more sequence diversity, further from the distribution learned during training)
* `omit_aas`: Amino acids to omit in designed regions
* `deterministic`: If True, enables reproducible results

In [ ]:
input_dir = "/content/drive/MyDrive/RFantibody_example_run/rfabd_test_out" # TODO: YOUR DIFFUSION OUTPUT DIR (from above)
output_dir = "/content/drive/MyDrive/RFantibody_example_run/mpnn_test_out" # TODO: DESIRED MPNN OUTPUT DIR (Google Drive recommended!!!)
loops = "H1,H2,H3,L1,L2,L3" # TODO: WHICH LOOPS TO DESIGN
seqs_per_struct = "2" # TODO: NUMBER OF SEQS TO DESIGN PER INPUT
temp = "0.1"
omit_aas = "CMX"
deterministic = False

Run MPNN. This should only take a second or so per sequence.

In [ ]:
# @title
mpnn_cmd = ["uv", "run", "proteinmpnn",
            "-i", input_dir, "-o", output_dir,
            "-l", loops, "-n", seqs_per_struct,
            "--omit-aas", omit_aas, "-t", temp]

if deterministic:
  mpnn_cmd.append("--deterministic")

p = subprocess.run(mpnn_cmd, cwd=REPO_DIR, text=True, capture_output=True)
print("STDOUT:\n", p.stdout)
print("STDERR:\n", p.stderr)

STDOUT:
 Running ProteinMPNN sequence design...
Input: /content/drive/MyDrive/RFantibody_example_run/rfabd_test_out
Loops: H1,H2,H3,L1,L2,L3
Sequences per structure: 2
Found GPU will run ProteinMPNN on GPU
Attempting pose: /content/drive/MyDrive/RFantibody_example_run/rfabd_test_out/rfabd_test_1.pdb
loopH: [26, 27, 28, 29, 30, 31, 32, 52, 53, 54, 55, 56, 57, 58, 100, 101, 102, 103, 104, 105, 106, 107, 108]
loopL: [143, 144, 145, 146, 147, 148, 149, 165, 166, 167, 168, 169, 170, 171, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215]
MPNN generated 2 sequences in 2 seconds
sequence_optimize: [('EVKLQESGGGLVQPGGSLKLSCATSGFDLSTKYMYWVRQTPEKRLEWVAYIAPQRPERTYYPDTVKGRFTISRDNAKNTLYLQMSRLKSEDTAMYYCARGSDPLHLDEWGQGTTVTVSSDIVLTQSPSSLSASLGDTITITCSGSGLFSWYQQKPGNIPKLLIYNTSTRAPGVPSRFSGSGSGTGFTLTISSLQPEDIATYYCSTWDSTTSNALHFGGGTKLEIK', 1.3800638), ('EVKLQESGGGLVQPGGSLKLSCATSGFDLSTKYMYWVRQTPEKRLEWVAYIAPSRPERTYYPDTVKGRFTISRDNAKNTLYLQMSRLKSEDTAMYYCARGDDPLNLSDWGQGTTVTVSSDIVLTQSPSSLSASLGDTITITCSGSGSF

In [ ]:
# @title
# check that it worked
if "Error" in p.stderr:
  print("ProteinMPNN failed to run. STDERR:")
  print(p.stderr)
else:
  print(f"ProteinMPNN succeeded. PDB files with designed sequences can be found in {output_dir}/*.pdb")

ProteinMPNN succeeded. PDB files with designed sequences can be found in /content/drive/MyDrive/RFantibody_example_run/mpnn_test_out/*.pdb


## C. Structure prediction
**The best practice is to use AlphaFold3 to predict the structure of your designed antibody in complex with the antigen.**

We recommend following instructions on the AlphaFold3 Github repo to get your own copy of the model up and running. There are cells below that will convert your MPNN outputs into json files that can be provided to AlphaFold3 inference.

You can also use DeepMind's web server to run 30 predictions a day, as long as your entire prediction is under 5000 amino acids long and does not contain banned amino acid sequences (e.g., viral proteins).

If neither of these options are accessible, continue to section D to run RoseTTAFold2 fine-tuned on antibodies.

### C.1. Generate json files for web server - TO EDIT

In [ ]:
# @title
%pip install biotite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 57.2/57.2 MB 48.8 MB/s eta 0:00:01
ERROR: Operation cancelled by user


In [ ]:
# @title
import os, glob
from biotite.structure import io
import biotite.structure as struc
import json

THREE_TO_ONE = {
    "ALA": "A",
    "ARG": "R",
    "ASN": "N",
    "ASP": "D",
    "CYS": "C",
    "GLN": "Q",
    "GLU": "E",
    "GLY": "G",
    "HIS": "H",
    "ILE": "I",
    "LEU": "L",
    "LYS": "K",
    "MET": "M",
    "PHE": "F",
    "PRO": "P",
    "SER": "S",
    "THR": "T",
    "TRP": "W",
    "TYR": "Y",
    "VAL": "V"
    }

def three_to_one(three):
  return THREE_TO_ONE[three]

def get_antibody_seqs_from_mpnn_pdb(pdb_file):
  """
  Scrape "H" (and "L") chain sequences from pdb files generated by ProteinMPNN.
  """
  seqs = {}
  aa = io.load_structure(pdb_file)
  # check chain IDs and only parse ones that at least have a chain labeled "H"
  chain_ids = struc.get_chains(aa).tolist()
  if "H" not in chain_ids:
    print(f"[WARNING] Skipping {pdb_file} as it doesn't contain a chain labeled 'H'")
    return None
  # parse chain sequences
  def _parse_chain(chid):
    sub_aa = aa[aa.chain_id == chid]
    _, res_names = struc.get_residues(sub_aa)
    return "".join([three_to_one(res) for res in res_names])

  h_seq = _parse_chain("H")
  seqs["H"] = h_seq

  if "L" in chain_ids:
    l_seq = _parse_chain("L")
    seqs["L"] = l_seq

  return seqs

def assemble_afserver_jsons(mpnn_out_dir, target_dict, out_path, seeds):
  # ensure that mpnn dir exists
  if not os.path.exists(mpnn_out_dir):
    print(f"[ERROR] mpnn output dir {mpnn_out_dir} not found")
    return
  # ensure that dir has pdb files in it
  mpnn_pdbs = glob.glob(os.path.join(mpnn_out_dir, "*.pdb"))
  if len(mpnn_pdbs) == 0:
    print(f"[ERROR] {mpnn_out_dir} does not contain any pdb files")
    return

  # create the outpath if it doesn't exist
  if not os.path.exists(out_path):
    os.makedirs(out_path)
  else:
    print(f"Found {out_path}, will deposit output jsons here")

  # make a separate json for each file
  for pdb_file in mpnn_pdbs:
    run_name = os.path.basename(pdb_file).replace(".pdb", "")
    out_file = os.path.join(out_path, f"{run_name}.json")
    out_dict = {"name": run_name, "modelSeeds": seeds, "sequences": [],
                "dialect": "alphafoldserver", "version": 1}
    with open(f"{out_file}", 'w') as f:
      ab_seqs = get_antibody_seqs_from_mpnn_pdb(pdb_file) # keys are "H" and "L"

      def _format_into_afserver(seq):
        """
        Takes a dictionary where chain IDs are keys and amino acid sequences are
        values and converts it into the appropriate format for an AF3 input json
        """
        chain_dict = {
            "proteinChain": {
                "sequence": "",
                "count": 1
            }
        }
        # Add chain id & sequence info to dictionary
        chain_dict["proteinChain"]["sequence"] += seq
        return chain_dict

      # add antibody H (and L, if it's there) to dictionary
      for _, seq in ab_seqs.items():
        out_dict["sequences"].append(_format_into_afserver(seq))
      # add target chains to dictionary
      for _, seq in target_dict.items():
        out_dict["sequences"].append(_format_into_afserver(seq))

      json.dump([out_dict], f, indent=2)
      print(f"Wrote {out_file}")


Edit the following cell to fill in your desired options:
* `mpnn_out_dir`: directory containing the PDBs output by running ProteinMPNN
* `target_dict`: dictionary where keys are target chains and values are the corresponding amino acid sequences for those chains. We recommend that you run these predictions with contiguous chains, if possible, for the highest accuracy. **NB: don't label any of these chains "H" or "L", or they'll conflict with your antibody!**
* `out_path`: path to directory where generated json files should be stored
* `seeds`: list containing random number (or numbers, if you want to run multiple seeds).

In [ ]:
mpnn_out_dir = "/content/drive/MyDrive/RFantibody_example_run/mpnn_test_out" # TODO: YOUR MPNN OUTPUT DIR (from above)
# TODO: YOUR TARGET CHIDS AND SEQUENCES (dict below)
target_dict = {"B": "GLFGAIAGFIENGWEGLIDGWYGFRHQNAQGEGTAADYKSTQSAIDQITGKLNRLIAKTNQQFELIDNEFNEVEKQIGNVINWTRDSITEVWSYNAELLIAMENQHTIDLADSEMDKLYERVKRQLRENAEEDGTGCFEIFHKCDDDCMASIRNNTYDHRKYREEAMQNRIQIDPVKLSSGYKDVILWFSFGASCFILLAIVMGLVFICVKNGNMRCTICI",
               "D": "GLFGAIAGFIENGWEGLIDGWYGFRHQNAQGEGTAADYKSTQSAIDQITGKLNRLIAKTNQQFELIDNEFNEVEKQIGNVINWTRDSITEVWSYNAELLIAMENQHTIDLADSEMDKLYERVKRQLRENAEEDGTGCFEIFHKCDDDCMASIRNNTYDHRKYREEAMQNRIQIDPVKLSSGYKDVILWFSFGASCFILLAIVMGLVFICVKNGNMRCTICI",
               "F": "GLFGAIAGFIENGWEGLIDGWYGFRHQNAQGEGTAADYKSTQSAIDQITGKLNRLIAKTNQQFELIDNEFNEVEKQIGNVINWTRDSITEVWSYNAELLIAMENQHTIDLADSEMDKLYERVKRQLRENAEEDGTGCFEIFHKCDDDCMASIRNNTYDHRKYREEAMQNRIQIDPVKLSSGYKDVILWFSFGASCFILLAIVMGLVFICVKNGNMRCTICI",
               "A": "DKICLGHHAVSNGTKVNTLTERGVEVVNATETVERTNIPRICSKGKRTVDLGQCGLLGTITGPPQCDQFLEFSADLIIERREGSDVCYPGKFVNEEALRQILRESGGIDKETMGFTYNGIRTNGVTSACKRSGSSFYAEMKWLLSNTDNAAFPQMTKSYKNTRKSPAIIVWGIHHSVSTAEQTKLYGSGNKLVTVGSSNYQQSFVPSPGARPQVNGLSGRIDFHWLILNPNDTVTFSFNGAFIAPDRASFLRGKSMGIQSGVQVDANCEGDCYHSGGTIISNLPFQNIDSRAVGKCPRYVKQRSLLLATGMKNVPEIPKGR",
               "C": "DKICLGHHAVSNGTKVNTLTERGVEVVNATETVERTNIPRICSKGKRTVDLGQCGLLGTITGPPQCDQFLEFSADLIIERREGSDVCYPGKFVNEEALRQILRESGGIDKETMGFTYNGIRTNGVTSACKRSGSSFYAEMKWLLSNTDNAAFPQMTKSYKNTRKSPAIIVWGIHHSVSTAEQTKLYGSGNKLVTVGSSNYQQSFVPSPGARPQVNGLSGRIDFHWLILNPNDTVTFSFNGAFIAPDRASFLRGKSMGIQSGVQVDANCEGDCYHSGGTIISNLPFQNIDSRAVGKCPRYVKQRSLLLATGMKNVPEIPKGR",
               "E": "DKICLGHHAVSNGTKVNTLTERGVEVVNATETVERTNIPRICSKGKRTVDLGQCGLLGTITGPPQCDQFLEFSADLIIERREGSDVCYPGKFVNEEALRQILRESGGIDKETMGFTYNGIRTNGVTSACKRSGSSFYAEMKWLLSNTDNAAFPQMTKSYKNTRKSPAIIVWGIHHSVSTAEQTKLYGSGNKLVTVGSSNYQQSFVPSPGARPQVNGLSGRIDFHWLILNPNDTVTFSFNGAFIAPDRASFLRGKSMGIQSGVQVDANCEGDCYHSGGTIISNLPFQNIDSRAVGKCPRYVKQRSLLLATGMKNVPEIPKGR"
               }
out_path = "/content/drive/MyDrive/RFantibody_example_run/afserver_jsons"  # TODO: DESIRED JSON OUT PATH
seeds = [784194]

In [ ]:
# @title
assemble_afserver_jsons(mpnn_out_dir, target_dict, out_path, seeds)

/usr/local/lib/python3.12/dist-packages/biotite/structure/io/pdb/file.py:475: UserWarning: 2116 elements were guessed from atom name
  warnings.warn(


Wrote /content/drive/MyDrive/RFantibody_example_run/afserver_jsons/rfabd_test_1_dldesign_0.json
Wrote /content/drive/MyDrive/RFantibody_example_run/afserver_jsons/rfabd_test_1_dldesign_1.json


/usr/local/lib/python3.12/dist-packages/biotite/structure/io/pdb/file.py:475: UserWarning: 2128 elements were guessed from atom name
  warnings.warn(


Wrote /content/drive/MyDrive/RFantibody_example_run/afserver_jsons/rfabd_test_2_dldesign_0.json
Wrote /content/drive/MyDrive/RFantibody_example_run/afserver_jsons/rfabd_test_2_dldesign_1.json


Download these json files and upload them to the server.

### C.2. Generate json files for commandline AlphaFold3 inference - TO EDIT
See the [AlphaFold3 github](https://github.com/google-deepmind/alphafold3) for instructions on installing the codebase and getting access to the weights.

In [ ]:
# @title
import os, glob
from biotite.structure import io
import biotite.structure as struc
import json

THREE_TO_ONE = {
    "ALA": "A",
    "ARG": "R",
    "ASN": "N",
    "ASP": "D",
    "CYS": "C",
    "GLN": "Q",
    "GLU": "E",
    "GLY": "G",
    "HIS": "H",
    "ILE": "I",
    "LEU": "L",
    "LYS": "K",
    "MET": "M",
    "PHE": "F",
    "PRO": "P",
    "SER": "S",
    "THR": "T",
    "TRP": "W",
    "TYR": "Y",
    "VAL": "V"
    }

def three_to_one(three):
  return THREE_TO_ONE[three]

def get_antibody_seqs_from_mpnn_pdb(pdb_file):
  """
  Scrape "H" (and "L") chain sequences from pdb files generated by ProteinMPNN.
  """
  seqs = {}
  aa = io.load_structure(pdb_file)
  # check chain IDs and only parse ones that at least have a chain labeled "H"
  chain_ids = struc.get_chains(aa).tolist()
  if "H" not in chain_ids:
    print(f"[WARNING] Skipping {pdb_file} as it doesn't contain a chain labeled 'H'")
    return None
  # parse chain sequences
  def _parse_chain(chid):
    sub_aa = aa[aa.chain_id == chid]
    _, res_names = struc.get_residues(sub_aa)
    return "".join([three_to_one(res) for res in res_names])

  h_seq = _parse_chain("H")
  seqs["H"] = h_seq

  if "L" in chain_ids:
    l_seq = _parse_chain("L")
    seqs["L"] = l_seq

  return seqs

def assemble_af3_jsons(mpnn_out_dir, target_dict, out_path, seeds):
  # ensure that mpnn dir exists
  if not os.path.exists(mpnn_out_dir):
    print(f"[ERROR] mpnn output dir {mpnn_out_dir} not found")
    return
  # ensure that dir has pdb files in it
  mpnn_pdbs = glob.glob(os.path.join(mpnn_out_dir, "*.pdb"))
  if len(mpnn_pdbs) == 0:
    print(f"[ERROR] {mpnn_out_dir} does not contain any pdb files")
    return

  # create the outpath if it doesn't exist
  if not os.path.exists(out_path):
    os.makedirs(out_path)
  else:
    print(f"Found {out_path}, will deposit output jsons here")

  # make a separate json for each file
  for pdb_file in mpnn_pdbs:
    run_name = os.path.basename(pdb_file).replace(".pdb", "")
    out_file = os.path.join(out_path, f"{run_name}.json")
    out_dict = {"name": run_name, "sequences": [], "modelSeeds": seeds, "dialect": "alphafold3", "version": 2}
    with open(f"{out_file}", 'w') as f:
      ab_seqs = get_antibody_seqs_from_mpnn_pdb(pdb_file) # keys are "H" and "L"

      def _format_into_af3(chid, seq, msas=True):
        """
        Takes a dictionary where chain IDs are keys and amino acid sequences are
        values and converts it into the appropriate format for an AF3 input json
        """
        chain_dict = {
            "protein": {
                "id": [],
                "sequence": ""
            }
        }
        # tell AF3 data pipeline not to look for MSAs
        if not msas:
          chain_dict["protein"]["unpairedMsa"] = ""
          chain_dict["protein"]["pairedMsa"] = ""
        # Add chain id & sequence info to dictionary
        chain_dict["protein"]["id"].append(chid)
        chain_dict["protein"]["sequence"] += seq
        return chain_dict

      # add antibody H (and L, if it's there) to dictionary
      for chid, seq in ab_seqs.items():
        out_dict["sequences"].append(_format_into_af3(chid, seq, msas=False))
      # add target chains to dictionary
      for chid, seq in target_dict.items():
        out_dict["sequences"].append(_format_into_af3(chid, seq, msas=True))

      json.dump(out_dict, f, indent=2)
      print(f"Wrote {out_file}")

Edit the following cell to fill in your desired options:
* `mpnn_out_dir`: directory containing the PDBs output by running ProteinMPNN
* `target_dict`: dictionary where keys are target chains and values are the corresponding amino acid sequences for those chains. We recommend that you run these predictions with contiguous chains, if possible, for the highest accuracy. **NB: don't label any of these chains "H" or "L", or they'll conflict with your antibody!**
* `out_path`: path to directory where generated json files should be stored
* `seeds`: list containing random number (or numbers, if you want to run multiple seeds).

In [ ]:
mpnn_out_dir = "/content/drive/MyDrive/RFantibody_example_run/mpnn_test_out" # TODO: YOUR MPNN OUTPUT DIR (from above)
# TODO: YOUR TARGET CHIDS AND SEQUENCES (dict below)
target_dict = {"A": "MELAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTNASLSFLQDIQEVQGYVLIAHNQVRQVPLQRLRIVRGTQLFEDNYALAVLDNGDPLNNTTPVTGASPGGLRELQLRSLTEILKGGVLIQRNPQLCYQDTILWKDIFHKNNQLALTLIDTNRSRACHPCSPMCKGSRCWGESSEDCQSLTRTVCAGGCARCKGPLPTDCCHEQCAAGCTGPKHSDCLACLHFNHSGICELHCPALVTYNTDTFESMPNPEGRYTFGASCVTACPYNYLSTDVGSCTLVCPLHNQEVTAEDGTQRCEKCSKPCARVCYGLGMEHLREVRAVTSANIQEFAGCKKIFGSLAFLPESFDGDPASNTAPLQPEQLQVFETLEEITGYLYISAWPDSLPDLSVFQNLQVIRGRILHNGAYSLTLQGLGISWLGLRSLRELGSGLALIHHNTHLCFVHTVPWDQLFRNPHQALLHTANRPEDECVGEGLACHQLCARGHCWGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYVNARHCLPCHPECQPQNGSVTCFGPEADQCVACAHYKDPPFCVARCPSGVKPDLSYMPIWKFPDEEGACQPCPINCTHSCVDLDDKGCPAEQRASPLTSIISAVVGILLVVVLGVVFGILIKRRQQKIRKYTMRRLLQEGGSENLYFQGGGSAQLEKELQALEKENAQLEWELQALEKELAQSNSLEVLFQ",
               "B": "MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNMYYENSYALAVLSNYDANKTGLKELPMRNLQEILHGAVRFSNNPALCNVESIQWRDIVSSDFLSNMSMDFQNHLGSCQKCDPSCPNGSCWGAGEENCQKLTKIICAQQCSGRCRGKSPSDCCHNQCAAGCTGPRESDCLVCRKFRDEATCKDTCPPLMLYNPTTYQMDVNPEGKYSFGATCVKKCPRNYVVTDHGSCVRACGADSYEMEEDGVRKCKKCEGPCRKVCNGIGIGEFKDSLSINATNIKHFKNCTSISGDLHILPVAFRGDSFTHTPPLDPQELDILKTVKEITGFLLIQAWPENRTDLHAFENLEIIRGRTKQHGQFSLAVVSLNITSLGLRSLKEISDGDVIISGNKNLCYANTINWKKLFGTSGQKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCVSCRNVSRGRECVDKCNLLEGEPREFVENSECIQCHPECLPQAMNITCTGRGPDNCIQCAHYIDGPHCVKTCPAGVMGENNTLVWKYADAGHVCHLCHPNCTYGCTGPGLEGCPTNGPKIPSIATGMVGALLLLLVVALGIGLFMRRRHIVRKRTLRRLLGGSENLYFQGGGSAAQLKKKLQALKKKNAQLKWKLQALKKKLAQSNSLEVLFQ"}
out_path = "/content/drive/MyDrive/RFantibody_example_run/af3_jsons" # TODO: DESIRED JSON OUT PATH
seeds = [784194]

In [ ]:
# @title
assemble_af3_jsons(mpnn_out_dir, target_dict, out_path, seeds)

Wrote /content/drive/MyDrive/RFantibody_example_run/af3_jsons/rfabd_test_1_dldesign_0.json
Wrote /content/drive/MyDrive/RFantibody_example_run/af3_jsons/rfabd_test_1_dldesign_1.json
Wrote /content/drive/MyDrive/RFantibody_example_run/af3_jsons/rfabd_test_2_dldesign_0.json
Wrote /content/drive/MyDrive/RFantibody_example_run/af3_jsons/rfabd_test_2_dldesign_1.json


/usr/local/lib/python3.12/dist-packages/biotite/structure/io/pdb/file.py:475: UserWarning: 2116 elements were guessed from atom name
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/biotite/structure/io/pdb/file.py:475: UserWarning: 2128 elements were guessed from atom name
  warnings.warn(


Download these files to a directory, then give that directory as input when running AF3 inference.

## D. RoseTTAFold2 - TO EDIT
The final step of the RFantibody pipeline is to use our antibody-finetuned RF2 to predict the structure of the sequences we just designed. We then assess whether RF2 is confident that the sequence will bind as we designed.

Fill out the cell below with the following options:
* `input_dir` Input directory of PDB files from running ProteinMPNN
* `output_dir`: Output directory to store PDB files of predictions
* `seed`: Random seed for reproducibility

In [ ]:
input_dir = "/content/drive/MyDrive/RFantibody_example_run/mpnn_test_out" # TODO: YOUR MPNN OUTPUT DIR (from above)
output_dir = "/content/drive/MyDrive/RFantibody_example_run/rf2_test_out" # TODO: DESIRED RF2 PREDICTION OUTPUT DIR
seed = "784194"

In [ ]:
# @title
rf2_cmd = ["uv", "run", "rf2",
           "-i", input_dir, "-o", output_dir,
           "-s", seed]

p = subprocess.run(rf2_cmd, cwd=REPO_DIR, text=True, capture_output=True)
print("STDOUT:\n", p.stdout)
print("STDERR:\n", p.stderr)

STDOUT:
 Running RF2 structure prediction...
Input: /content/drive/MyDrive/RFantibody_example_run/mpnn_test_out
Recycles: 10
Hotspot show proportion: 0.1
Running RF2 with the following configs: {'input': {'pdb': None, 'pdb_dir': '/content/drive/MyDrive/RFantibody_example_run/mpnn_test_out', 'quiver': None}, 'output': {'pdb_dir': '/content/drive/MyDrive/RFantibody_example_run/rf2_test_out', 'quiver': None, 'output_intermediates': False}, 'inference': {'num_recycles': 10, 'hotspot_show_proportion': 0.1, 'cautious': True, 'seed': 784194}, 'model': {'model_weights': '/content/drive/MyDrive/RFantibody_cache/weights/RF2_ab.pt'}, 'model_param': {'n_extra_block': 4, 'n_main_block': 36, 'd_msa': 256, 'd_pair': 128, 'd_templ': 64, 'n_head_msa': 8, 'n_head_pair': 4, 'n_head_templ': 4, 'd_hidden': 32, 'd_hidden_templ': 32, 'd_t1d': 23, 'p_drop': 0.0, 'SE3_param_full': {'num_layers': 1, 'num_channels': 48, 'num_degrees': 2, 'l0_in_features': 32, 'l0_out_features': 32, 'l1_in_features': 2, 'l1_out_f

In [ ]:
# @title
# check that it worked
if "Error" in p.stderr:
  print("[ERROR] RF2 did not run successfully. STDERR:")
  print(p.stderr)
else:
  print(f"RF2 ran successfully! Predictions can be found in {output_dir}")

RF2 ran successfully! Predictions can be found in /content/drive/MyDrive/RFantibody_example_run/rf2_test_out
